In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error
from math import sqrt

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving BDO Historical Data.csv to BDO Historical Data.csv


In [ ]:
DATA_PATH = "BDO Historical Data.csv"
PRED_DAYS = 10
SUB_OUT = "submission.csv"

In [ ]:
df = pd.read_csv("BDO Historical Data.csv")

display(df.head())

,Date,Price,Open,High,Low,Vol.,Change %
0,07/31/2025,142.7,147.2,147.6,142.1,5.67M,-2.93%
1,07/30/2025,147.0,149.0,149.0,147.0,3.47M,-1.01%
2,07/29/2025,148.5,149.4,149.9,148.5,2.56M,-0.34%
3,07/28/2025,149.0,151.5,151.5,147.8,3.01M,-2.10%
4,07/25/2025,152.2,153.9,153.9,151.5,1.64M,-1.10%


In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format="%m/%d/%Y")
df.sort_values('Date', inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
df = df[['Date', 'Open', 'High', 'Low', 'Price']].copy()

In [ ]:
for col in ['Open', 'High', 'Low', 'Price']:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')

In [ ]:
for lag in [1, 2, 3, 5, 7, 10]:
    df[f'Price_lag{lag}'] = df['Price'].shift(lag)

In [ ]:
for window in [3, 5, 7, 10]:
    df[f'Price_roll_mean{window}'] = df['Price'].rolling(window).mean()

In [ ]:
df['Price_ewm_span10'] = df['Price'].ewm(span=10, adjust=False).mean()

In [ ]:
df['HL_range'] = df['High'] - df['Low']

In [ ]:
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
train = df[:-PRED_DAYS]
test = df[-PRED_DAYS:]
X = train.drop(['Date', 'Price'], axis=1)
y = train['Price']
X_test = test.drop(['Date', 'Price'], axis=1)

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
param_grid = {
    'n_estimators': [200, 500, 1000],
    'max_depth': [5, 10, None],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt']
}
base_rf = RandomForestRegressor(random_state=42, n_jobs=-1)
grid = GridSearchCV(
    estimator=base_rf,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    verbose=1
)
grid.fit(X, y)
best_rf = grid.best_estimator_
print("Best parameters found:", grid.best_params_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
135 fits failed out of a total of 270.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
135 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
  File "/usr/local/lib/python3.11/dist-packages/sklearn/base.py", line 436, in _validate_params
    validate_parameter_constraints(
  File "/usr/local/lib/python3.11/dist-packages/sklearn/util

Best parameters found: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 1000}


In [ ]:
val_size = int(len(X) * 0.2)
X_train2, X_val = X.iloc[:-val_size], X.iloc[-val_size:]
y_train2, y_val = y.iloc[:-val_size], y.iloc[-val_size:]
best_rf.fit(X_train2, y_train2)
val_preds = best_rf.predict(X_val)
val_rmse = sqrt(mean_squared_error(y_val, val_preds))
print(f"Validation RMSE (last 20%): {val_rmse:.4f}")

Validation RMSE (last 20%): 1.5869


In [ ]:
preds = best_rf.predict(X_test)
test['PredictedPrice'] = preds
export_df = test[['Date', 'PredictedPrice']]
export_df.to_csv(SUB_OUT, index=False)
print(f"✅ {SUB_OUT} written ({os.path.getsize(SUB_OUT)/1024:.1f} KB)")

✅ submission.csv written (0.3 KB)


/tmp/ipython-input-1636358406.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['PredictedPrice'] = preds


In [ ]:
files.download(SUB_OUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>